https://www.mssqltips.com/sqlservertip/6967/databricks-delta-change-data-feed/

In [0]:
CREATE DATABASE IF NOT EXISTS cdc;
DROP TABLE IF EXISTS cdc.OrdersSilver
DROP TABLE IF EXISTS cdc.OrdersGold

In [0]:
CREATE TABLE cdc.OrdersSilver (
OrderID int,
UnitPrice int,
Quantity int,
Customer string
)
USING DELTA
LOCATION "/mnt/raw/OrdersSilver"
TBLPROPERTIES (delta.enableChangeDataFeed = true);

In [0]:
CREATE TABLE cdc.OrdersGold (
OrderID int,
OrderTotal int,
Customer string
)
USING DELTA
LOCATION "/mnt/raw/OrdersGold"

In [0]:
INSERT INTO cdc.OrdersSilver
SELECT 1 OrderID, 96 as UnitPrice, 5 as Quantity, "A" as Customer
UNION
SELECT 2 OrderID, 450 as UnitPrice, 10 as Quantity, "B" as Customer
UNION
SELECT 3 OrderID, 134 as UnitPrice, 7 as Quantity, "C" as Customer
UNION
SELECT 4 OrderID, 847 as UnitPrice, 8 as Quantity, "D" as Customer
UNION
SELECT 5 OrderID, 189 as UnitPrice, 15 as Quantity, "E" as Customer;
 
SELECT * FROM cdc.OrdersSilver

In [0]:
INSERT INTO cdc.OrdersGold
SELECT OrderID, UnitPrice * Quantity AS OrderTotal, Customer FROM cdc.OrdersSilver;
 
SELECT * FROM cdc.OrdersGold

Change Data Capture


In [0]:
CREATE OR REPLACE TEMPORARY VIEW latest_version as
SELECT * 
    FROM 
         (SELECT *, rank() over (partition by OrderID order by _commit_version desc) as rank
          FROM table_changes('cdc.OrdersSilver', 2, 5)
          WHERE _change_type !='update_preimage')
    WHERE rank=1

In [0]:
UPDATE cdc.OrdersSilver SET Quantity = 20 WHERE OrderID = 1;
DELETE FROM cdc.OrdersSilver WHERE Customer = 'D';
INSERT INTO cdc.OrdersSilver SELECT 6 OrderID, 100 as UnitPrice, 10 as Quantity, "F" as Customer;
 
SELECT * FROM table_changes('cdc.OrdersSilver', 2, 5) order by _commit_timestamp;

In [0]:
MERGE INTO cdc.OrdersGold og USING latest_version os ON og.OrderID = os.OrderID
WHEN MATCHED AND os._change_type='update_postimage' THEN UPDATE SET OrderTotal = os.UnitPrice * os.Quantity      
WHEN MATCHED AND os._change_type='delete' THEN DELETE 
WHEN NOT MATCHED THEN INSERT (OrderID, OrderTotal, Customer) VALUES (os.OrderID, os.UnitPrice * os.Quantity, os.Customer)

In [0]:
DROP TABLE cdc.OrdersSilver;
DROP TABLE cdc.OrdersGold;